In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, explode
from pyspark.sql import functions as F 

In [2]:
try:
   spark.stop()
except:
     pass 

spark = SparkSession.builder\
        .appName('traitement_NLP').getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/12 23:19:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [8]:
# version de la version de spark 
spark.version

'3.5.7'

In [9]:
# lecture de nos fichier hdfs 
df = spark.read.option("mergeSchema", "true").parquet("hdfs://namenode:8020/ben/dataLake/")
df.count()

8266

In [10]:
df = df.dropDuplicates()
# compter les nombres de lignes qu'on a 
df.count()

462

In [11]:
# voir le schema ou les differents colonnes de nos données 
df.printSchema()

root
 |-- entreprise: string (nullable = true)
 |-- poste: string (nullable = true)
 |-- niveau_etude: string (nullable = true)
 |-- niveau_experience: string (nullable = true)
 |-- contrat_propose: string (nullable = true)
 |-- region: string (nullable = true)
 |-- Competence: string (nullable = true)
 |-- date_de_publication: string (nullable = true)
 |-- formation: string (nullable = true)
 |-- lien: string (nullable = true)
 |-- contract_propose: string (nullable = true)



In [12]:
# Les nombres de differents poste et  entreprises existantes 
df.select("poste").distinct().count()

437

In [13]:
df.select("entreprise").distinct().count()

#----------------- *** Colonnes a nettoyer et organiser ***-----------------------#
# Apres un passage de select, exemple la colonne suivant '"entreprise"
df.select("entreprise").distinct().show(5, truncate=False)

+------------------+
|entreprise        |
+------------------+
|Plan International|
|Adm Value Sénégal |
|CEVA Logistics    |
|Kuikdel           |
|TECTRA SÉNÉGAL    |
+------------------+
only showing top 5 rows



In [14]:
###------------- traitement de chaque colonne -------------------------------- ###### 

# ---------------- pour la colonne formation 

df_formation = df.withColumn(
    "formation_clean",
    F.trim(
        F.regexp_replace(F.col("formation"), r"</li><li>", ", "))
)

df_formation = df_formation.withColumn(
    "formation_clean",
    F.regexp_replace(F.col("formation_clean"), r"</?li>", "")
)

df_formation = df_formation.withColumn(
    "formation_clean",
    F.regexp_replace(F.col("formation_clean"), r"[\n]", "")
)

df_formation = df_formation.withColumn(
    "formation_clean",
    F.regexp_replace(F.col("formation_clean"),r"</?strong>", "")
)

df_formation = df_formation.withColumn(
    "formation_clean",
    F.trim(F.regexp_replace(F.col("formation_clean"), r"<[^>]+>", ""))
)

df_formation.drop("formation")

DataFrame[entreprise: string, poste: string, niveau_etude: string, niveau_experience: string, contrat_propose: string, region: string, Competence: string, date_de_publication: string, lien: string, contract_propose: string, formation_clean: string]

In [15]:
#-------------- Pour la colonne niveau_etude--------------

df_etude = df_formation.withColumn(
    "niveau_etude_clean",
    split(col("niveau_etude"), r"\s*(?:,|&|et)\s*")
)

df_etude = df_etude.drop("niveau_etude")
df_etude = df_etude.drop("formation")

In [16]:
##------------ contract proposé ---------------------------
df_contract = df_etude.withColumn(
    "contract",
    split(col("contrat_propose"), r"\s*(?:,|&|et)\s*")
)

df_contract = df_contract.drop("contrat_propose")


In [17]:
### ------------- Pour la region 

df_region = df_contract.withColumn(
    "region_clean",
    F.trim(F.regexp_replace(F.col("region"), r"\s*(?:\n|\t)", ""))

)

df_region = df_region.drop("region")

In [18]:
# ----- experience 

df_experience = df_region.withColumn(
    "experience", 
    split(col("niveau_experience"), r"\s*(?:,|&|-)\s*")
)

df_experience = df_experience.drop("niveau_experience")


In [19]:
#---------- Traitement des competences 

df_competence = df_experience.withColumn(
    "competences",
    split(col("Competence"), r"\s*(-)\s*")
)


In [20]:
###---------------------------------------------------------------------------------#######""
#------------------------------------------------------------------------------------#
# Creation de la premiere dataset qui contient toutes les données 

df_clean = df_competence.drop("Competence")


df_clean.show(5, truncate=False)

+----------------+-----------------------------------------------------+-------------------+------------------------------------------------------------------------------------------------------------------------------------+----------------+-----------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------+----------------+--------------------------+-----------------------------------------+--------------------------------------------------------------------------------------------------------+
|entreprise      |poste                                                |date_de_publication|lien                                                                                                                                |contract_propose|formation_clean                                                                                                              |

In [21]:
print("dataset normale sur les offres")
df_clean.show(10, truncate=False)

dataset normale sur les offres
+---------------------+--------------------------------------------------------------------+----------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+-----------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------+----------------------+-----------------------------------------------------------------+-----------------------------------------+--------------------------------------------------------------------------------------------------------+
|entreprise           |poste                                                               |date_de_publication   |lien                                                                                                                               

In [22]:
#### ---------------- Creation du dataSet normale -------------------------###


##########___-------Mise en forme----------__________________________###
df_propres = df_clean.select("entreprise", "poste", col("competences").alias("competence"),
col("formation_clean").alias("formation"), col("niveau_etude_clean").alias("niveau_etude"),
col("contract").alias("contrat"), col("experience").alias("experience"),
col("region_clean").alias("region"), col("date_de_publication").alias("date_de_pulication")
)


In [23]:
# Creation du deuxieme dataset pour le machine learning

df_ml = df_clean.select("entreprise", "poste", explode(col("competences")).alias("competence"),
col("formation_clean").alias("formation"), explode(col("niveau_etude_clean")).alias("niveau_etude"),
explode(col("contract")).alias("contrat"), explode(col("experience")).alias("experience"),
col("region_clean").alias("region"), col("date_de_publication").alias("date_de_pulication")
)


print("dataset pour le machine learning et l'analyse analytique")
df_ml.show(10, truncate=False)

dataset pour le machine learning et l'analyse analytique
+----------------+-----------------------------------+--------------+-------------------------------------+-----------------------+-------+---------------------+-------------+------------------+
|entreprise      |poste                              |competence    |formation                            |niveau_etude           |contrat|experience           |region       |date_de_pulication|
+----------------+-----------------------------------+--------------+-------------------------------------+-----------------------+-------+---------------------+-------------+------------------+
|INTRA INTERIM SN|Ouvrier d’Usine - Roumanie (Europe)|Accompagnement|Niveau d’anglais intermédiaire requis|Qualification avant bac|CDI    |Etudiant             |International|20.06.2025        |
|INTRA INTERIM SN|Ouvrier d’Usine - Roumanie (Europe)|Accompagnement|Niveau d’anglais intermédiaire requis|Qualification avant bac|CDI    |jeune diplômé et plus|In

In [24]:

### ---------------------- nos liens --------------------------------------------####
url = "jdbc:postgresql://postgres_warehouse:5432/datawarehouse"
user = "admin"
password = "admin_pwd"
driver = "org.postgresql.Driver"


In [26]:
df_propres.write\
    .format("jdbc")\
    .option("url", url)\
    .option("dbtable","offres_emploi")\
    .option("user", user)\
    .option("password",password)\
    .option("driver", driver)\
    .mode("append")\
    .save()

In [29]:
###---------------Creation du dataset ML --------------------------------####

df_ml.write\
    .format("jdbc")\
    .option("url", url)\
    .option("dbtable","offres_emploi_ml")\
    .option("user", user)\
    .option("password",password)\
    .option("driver", driver)\
    .mode("append")\
    .save()

In [30]:
print("--------------------------tout esst carree-----------------------------")


spark.stop()

del spark
print('la session spark est arretée et supprimée')

--------------------------tout esst carree-----------------------------
la session spark est arretée et supprimée


26/04/13 00:13:53 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-611b07fc-c2e8-49ce-881c-a92df7861a0d/pyspark-7889ab68-02b7-4179-978b-3aabf307feb3. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-611b07fc-c2e8-49ce-881c-a92df7861a0d/pyspark-7889ab68-02b7-4179-978b-3aabf307feb3
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:174)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:109)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:90)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1126)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4(ShutdownHookManager.scala:65)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun

In [ ]:


# creation d'une session spark 
spark = SparkSession.builder \
    .appName("Traitement_depuis_hdfs") \
    .master("yarn") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020") \
    .config("spark.hadoop.yarn.resourcemanager.hostname", "resourcemanager") \
    .getOrCreate()

# version de la version de spark 
spark.version

# lecture de nos fichier hdfs 
df = spark.read.option("mergeSchema", "true").parquet("hdfs://namenode:8020/ben/dataLake/")
df = df.dropDuplicates()
# compter les nombres de lignes qu'on a 
df.count()

# voir le schema ou les differents colonnes de nos données 
df.printSchema()

# >>> df.printSchema()
# root
#  |-- entreprise: string (nullable = true)
#  |-- poste: string (nullable = true)
#  |-- niveau_etude: string (nullable = true)
#  |-- niveau_experience: string (nullable = true)
#  |-- contrat_propose: string (nullable = true)
#  |-- region: string (nullable = true)
#  |-- Competence: string (nullable = true)
#  |-- date_de_publication: string (nullable = true)
#  |-- formation: string (nullable = true)

# Les nombres de differents poste et  entreprises existantes 
df.select("poste").distinct().count()

df.select("entreprise").distinct().count()

#----------------- *** Colonnes a nettoyer et organiser ***-----------------------#
# Apres un passage de select, exemple la colonne suivant '"entreprise"
df.select("entreprise").distinct().show(5, truncate=False)



###---------------µ*******************************************----------------###

####     niveau_etude , niveau_experience, contrat_propose
####     region, Competence, formation

###------------- traitement de chaque colonne -------------------------------- ###### 

# ---------------- pour la colonne formation 

df_formation = df.withColumn(
    "formation_clean",
    F.trim(
        F.regexp_replace(F.col("formation"), r"</li><li>", ", "))
)

df_formation = df_formation.withColumn(
    "formation_clean",
    F.regexp_replace(F.col("formation_clean"), r"</?li>", "")
)

df_formation = df_formation.withColumn(
    "formation_clean",
    F.regexp_replace(F.col("formation_clean"), r"[\n]", "")
)

df_formation = df_formation.withColumn(
    "formation_clean",
    F.regexp_replace(F.col("formation_clean"),r"</?strong>", "")
)

df_formation = df_formation.withColumn(
    "formation_clean",
    F.trim(F.regexp_replace(F.col("formation_clean"), r"<[^>]+>", ""))
)

df_formation.drop("formation")

#-------------- Pour la colonne niveau_etude--------------

df_etude = df_formation.withColumn(
    "niveau_etude_clean",
    split(col("niveau_etude"), r"\s*(?:,|&|et)\s*")
)

df_etude = df_etude.drop("niveau_etude")
df_etude = df_etude.drop("formation")

##------------ contract proposé ---------------------------
df_contract = df_etude.withColumn(
    "contract",
    split(col("contrat_propose"), r"\s*(?:,|&|et)\s*")
)

df_contract = df_contract.drop("contrat_propose")

### ------------- Pour la region 

df_region = df_contract.withColumn(
    "region_clean",
    F.trim(F.regexp_replace(F.col("region"), r"\s*(?:\n|\t)", ""))

)

df_region = df_region.drop("region")

# ----- experience 

df_experience = df_region.withColumn(
    "experience", 
    split(col("niveau_experience"), r"\s*(?:,|&|-)\s*")
)

df_experience = df_experience.drop("niveau_experience")


#---------- Traitement des competences 

df_competence = df_experience.withColumn(
    "competences",
    split(col("Competence"), r"\s*(-)\s*")
)

###---------------------------------------------------------------------------------#######""
#------------------------------------------------------------------------------------#
# Creation de la premiere dataset qui contient toutes les données 

df_clean = df_competence.drop("Competence")


df_clean.show(5, truncate=False)

###---------------------------------------------------------------------------------#######""
#------------------------------------------------------------------------------------#
# Creation du deuxieme dataset pour le machine learning

df_ml = df_clean.select("entreprise", "poste", explode(col("competences")).alias("competence"),
col("formation_clean").alias("formation"), explode(col("niveau_etude_clean")).alias("niveau_etude"),
explode(col("contract")).alias("contrat"), explode(col("experience")).alias("experience"),
col("region_clean").alias("region"), col("date_de_publication").alias("date_de_pulication")
)


print("dataset pour le machine learning et l'analyse analytique")
df_ml.show(10, truncate=False)

print("dataset normale sur les offres")
df_clean.show(10, truncate=False)



####--------________---------------========------_______
#------------------------ Datawarehouse --------------------------------------####
#####-------_____============_______

print("Ingection vers notre dataWarehouse")


### ---------------------- nos liens --------------------------------------------####
url = "jdbc:postgresql://postgres_warehouse:5432/ DataWarehouse_JOB"
user = "admin"
password = "admin_pwd"
driver = "org.postgresql.Driver"


###---------------Creation du dataset ML --------------------------------####

df_ml.write\
    .format("jdbc")\
    .option("url", url)\
    .option("dbtable","offres_emploi_ml")\
    .option("user", user)\
    .option("password",password)\
    .option("driver", driver)\
    .mode("append")\
    .save()


#### ---------------- Creation du dataSet normale -------------------------###


##########___-------Mise en forme----------__________________________###
df_propres = df_clean.select("entreprise", "poste", col("competences").alias("competence"),
col("formation_clean").alias("formation"), col("niveau_etude_clean").alias("niveau_etude"),
col("contract").alias("contrat"), col("experience").alias("experience"),
col("region_clean").alias("region"), col("date_de_publication").alias("date_de_pulication")
)


df_propres.write\
    .format("jdbc")\
    .option("url", url)\
    .option("dbtable","offres_emploi")\
    .option("user", user)\
    .option("password",password)\
    .option("driver", driver)\
    .mode("append")\
    .save()


print("--------------------------tout esst carree-----------------------------")


spark.stop()

del spark
print('la session spark est arretée et supprimée')


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/08 13:29:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/08 13:29:58 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
26/04/08 13:30:38 WARN TaskSetManager: Lost task 1.0 in stage 0.0 (TID 1) (nodemanager executor 1): org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.util.ThreadUtils$.parmap(ThreadUtils.scala:387)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.readParquetFootersInParallel(ParquetFileFormat.scala:443)
	at org.apache.spark.sql.execution.datasources.parquet.Parquet